In [5]:
#MCP

import sys
import asyncio

# Fix for Windows issues in Jupyter notebooks
if sys.platform == "win32":
    # 1. Use ProactorEventLoop for subprocess support
    if not isinstance(asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy):
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

    # 2. Redirect stderr to avoid fileno() error when launching MCP servers
    if "ipykernel" in sys.modules:
        sys.stderr = sys.__stderr__

In [6]:
#1.stdio通信
#Time MCP服务

# 1.加载环境变量
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage
from langchain_mcp_adapters.client import MultiServerMCPClient

load_dotenv()

# 2.初始化模型
model = init_chat_model(
    model="deepseek-v4-flash",
    extra_body={"thinking": {"type": "disabled"}}
)

# 3.定义工具，用MCP获取工具
# 3.1.定义mcp client
client = MultiServerMCPClient(
    {
       "time": {
      "transport": "stdio",
      "command": "uvx",
      "args": [
        "mcp-server-time",
        "--local-timezone=Asia/Shanghai"
      ]
    } 
    }
)

# 3.2.用client拉取tool
# 注意MultiServerMCPClient是异步的，结果都是协程对象，需要await
tools= await client.get_tools()

In [7]:
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

#初始化模型
model = init_chat_model(
    model="deepseek-v4-flash",
    extra_body={"thinking": {"type": "disabled"}}
)

#创建agent
agent = create_agent(
            model= model,
            tools= tools,
)

#调用agent
response =  await agent.ainvoke({"messages":[HumanMessage("常州今天时间是多少")]})
for message in response["messages"]:
    message.pretty_print()


================================ Human Message =================================

常州今天时间是多少
================================== Ai Message ==================================
Tool Calls:
  get_current_time (call_00_jA4dP7Vbaotky8u7fYTs3175)
 Call ID: call_00_jA4dP7Vbaotky8u7fYTs3175
  Args:
    timezone: Asia/Shanghai
================================= Tool Message =================================
Name: get_current_time

[{'type': 'text', 'text': '{\n  "timezone": "Asia/Shanghai",\n  "datetime": "2026-10-27T14:08:44+08:00",\n  "day_of_week": "Tuesday",\n  "is_dst": false\n}', 'id': 'lc_7e2de6ae-3eff-481b-9a04-b3afe9f115fd'}]
================================== Ai Message ==================================

常州（中国标准时间，UTC+8）现在是：

**2026年10月27日（星期二）14:08**

需要换算成其他时区的时间吗？


In [8]:
#2.http MCP
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

# 连接Kiwi MCP服务器
client = MultiServerMCPClient(
    {
        "kiwi": {
            "transport": "http",
            "url": "https://mcp.kiwi.com",
        },
        "time": {
              "transport": "stdio",
              "command": "uvx",
              "args": [
                "mcp-server-time",
                "--local-timezone=Asia/Shanghai"
              ]
            } 
    }
)

#注意：MultiServerMCPClient是异步的，结果都是协程对象，需要await
tools= await client.get_tools()

In [9]:
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

model = init_chat_model(
    model="deepseek-v4-flash",
    extra_body={"thinking": {"type": "disabled"}}
)

agent = create_agent(
            model= model,
            tools= tools,
            system_prompt="You are a travel agent. Help user find best flights. No follow up questions. If the user uses Chinese, use zh-cn as the locale value.",
)

response = await agent.ainvoke({"messages":[HumanMessage("查一下下周二从上海到北京的航班")]})
for message in response["messages"]:
    message.pretty_print()


================================ Human Message =================================

查一下下周二从上海到北京的航班
================================== Ai Message ==================================
Tool Calls:
  get_current_time (call_00_oezj0NrGev0CFQRvpmFS3600)
 Call ID: call_00_oezj0NrGev0CFQRvpmFS3600
  Args:
    timezone: Asia/Shanghai
================================= Tool Message =================================
Name: get_current_time

[{'type': 'text', 'text': '{\n  "timezone": "Asia/Shanghai",\n  "datetime": "2026-10-27T14:08:59+08:00",\n  "day_of_week": "Tuesday",\n  "is_dst": false\n}', 'id': 'lc_002faaf2-748c-473b-b495-3de3e771a8db'}]
================================== Ai Message ==================================
Tool Calls:
  search-flight (call_00_blm9JjKwOxvexGct2vGC3913)
 Call ID: call_00_blm9JjKwOxvexGct2vGC3913
  Args:
    flyFrom: 上海
    flyTo: 北京
    departureDate: 03/11/2026
    locale: zh-cn
    currency: CNY
================================= Tool Message =========================

In [ ]:
#自定义MCP
from fastmcp import FastMCP

#初始化mcp
mcp = FastMCP("Math")

@mcp.tool()
def add(a:float, b:float)->float:
    """Add two numbers together"""
    return a + b

@mcp.tool()
def multiple(a:float, b:float)->float:
    """Multiply two numbers together"""
    return a * b

#启动mcp
if __name__ == "__main__":
    mcp.run(transport="stdio")

In [11]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

# 连接自定义MCP服务
client = MultiServerMCPClient(
    {
        "math": {
            "transport": "stdio",
            "command": "python",
            "args": ["3.3-math_mcp_server.py"] # 这里是脚本所在的路径，此处是相对路径
        }
    }
)
# 获取工具
tools = await client.get_tools()

# 创建agent
agent = create_agent(
    model="deepseek-chat",
    tools=tools,
    system_prompt="You are a helpful agent. You must use tools to answer math question."
)

# 调用测试
response = await agent.ainvoke({
    "messages": [HumanMessage("467和529的平方根之和是多少?")]
})

for message in response["messages"]:
    message.pretty_print()

================================ Human Message =================================

467和529的平方根之和是多少?
================================== Ai Message ==================================

计算√467 + √529。

√529 = 23（因为23² = 529）。

计算√467：
Tool Calls:
  square_root (call_00_ER27GnXNkbEH6kJgYebk4835)
 Call ID: call_00_ER27GnXNkbEH6kJgYebk4835
  Args:
    x: 467
================================= Tool Message =================================
Name: square_root

[{'type': 'text', 'text': '21.61018278497431', 'id': 'lc_6552045b-b780-4678-bf97-e88028eda911'}]
================================== Ai Message ==================================
Tool Calls:
  add (call_00_pyTSPGq0Bd9S7VGgzw096458)
 Call ID: call_00_pyTSPGq0Bd9S7VGgzw096458
  Args:
    a: 21.61018278497431
    b: 23
================================= Tool Message =================================
Name: add

[{'type': 'text', 'text': '44.61018278497431', 'id': 'lc_fda6e854-ceb0-4df9-a778-b371b14a5f2b'}]
================================== Ai Me